# Real-checkpoint component and cached-trajectory diagnostic

Enable Internet and T4 GPU, then Run All in a fresh Kaggle session.
Exact source and the frozen environment are embedded; no dataset or Git push is needed.

The four cells set up/verify the pinned checkpoint, run CPU checks, run the T4
diagnostic and export one ZIP. Download it even on failure.

Use 15 fixture tokens then feed 32 fixed ground-truth continuation IDs. Both paths
see identical histories and attend to lengths 16 through 47. This exercises actual
KV reuse and page boundaries; it is not greedy generation or a quality dataset.
Record complete logit/KV errors at all steps without discarding numerical failures.

At steps 0, 15 and 31, inspect both paths' real-weight RMSNorm, SwiGLU and final-head
operations against separately written FP64 reductions with documented FP16 rounding
boundaries. Save inputs and native/reference outputs. These are diagnostic errors,
not new pass/fail limits or a full independent model oracle. Residual composition
has sparse CPU coverage; this does not independently validate its real-weight math.

Run once without instrumentation and once with it. Both paths' logits must be exact
between runs at every step, or stop interpretation and retain the failure.
Only GPU 0 is visible. The full T4 runtime and end-to-end behavior of this new
collector remain unvalidated until its returned ZIP is reviewed.

This does not change serving arithmetic, thresholds or the comparator; P5.3 stays
open. vLLM quality comparison and long-context coverage up to 2048 remain future work.


In [ ]:
import base64
import hashlib
import io
import json
import os
import subprocess
import sys
import tempfile
import traceback
import zipfile
from pathlib import Path

from IPython.display import FileLink, display

PAYLOAD = "..."
SNAPSHOT_SHA256 = "b7a3b052bbb0f4f3860ffc8a44c41718478071f4f941f2b8e15efba0af6d811d"


output_override = os.environ.get("PAGEDCORE_EVIDENCE_WORKDIR")
working = (
    Path(output_override).expanduser().resolve()
    if output_override
    else Path("/kaggle/working")
    if Path("/kaggle/working").exists()
    else Path.cwd()
)
working.mkdir(parents=True, exist_ok=True)
run_root = Path(tempfile.mkdtemp(prefix="pagedcore-checkpoint-", dir=working))
source = run_root / "source"
evidence = run_root / "evidence"
evidence.mkdir()
payload = base64.b64decode(PAYLOAD)
assert hashlib.sha256(payload).hexdigest() == SNAPSHOT_SHA256
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    archive.extractall(source)
identity = json.loads((source / "source-identity.json").read_text())
environment = dict(
    os.environ,
    CUDA_VISIBLE_DEVICES="0",
    CUBLAS_WORKSPACE_CONFIG=":4096:8",
    OMP_NUM_THREADS="1",
    MKL_NUM_THREADS="1",
    TOKENIZERS_PARALLELISM="false",
)
commands = []


def run(name, command, *, required=False):
    record = {"name": name, "argv": list(map(str, command)), "cwd": str(source)}
    commands.append(record)
    print(name, flush=True)
    with (evidence / (name + ".log")).open("w") as log:
        process = subprocess.Popen(
            command, cwd=source, env=environment, stdout=log, stderr=subprocess.STDOUT
        )
        try:
            while process.poll() is None:
                try:
                    process.wait(timeout=30)
                except subprocess.TimeoutExpired:
                    print(name + " is running; progress is saved in " + str(log.name), flush=True)
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=15)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            record["interrupted"] = True
            raise
    record["returncode"] = process.returncode
    (evidence / "commands.json").write_text(json.dumps(commands, indent=2))
    print(name, "exit", process.returncode, flush=True)
    if required and process.returncode:
        raise RuntimeError(name + " failed; inspect its retained log")


def verify_source():
    checks = {
        name: hashlib.sha256((source / name).read_bytes()).hexdigest()
        for name in identity["sha256"]
    }
    assert checks == identity["sha256"], "source changed after notebook generation"
    return checks


setup_ok = False
try:
    verify_source()
    run("setup-uv", [sys.executable, "-m", "pip", "install", "uv==0.11.27"], required=True)
    run(
        "setup-frozen",
        [
            sys.executable,
            "-m",
            "uv",
            "sync",
            "--frozen",
            "--extra",
            "cuda",
            "--extra",
            "dev",
            "--python",
            (source / ".python-version").read_text().strip(),
        ],
        required=True,
    )
    python = source / ".venv/bin/python"
    run(
        "resolved-dependencies",
        [sys.executable, "-m", "uv", "pip", "freeze", "--python", str(python)],
    )
    run("gpu-inventory", ["nvidia-smi", "-q"])

    target_code = "import json,sys,torch; from pathlib import Path; from pagedcore.cli import collect_environment_diagnostics; assert torch.cuda.is_available() and torch.cuda.device_count()==1; assert 'T4' in torch.cuda.get_device_name(0); assert torch.__version__=='2.7.1+cu126'; p=torch.cuda.get_device_properties(0); Path(sys.argv[1]).write_text(json.dumps(dict(torch=torch.__version__,name=p.name,uuid=str(p.uuid),capability=[p.major,p.minor],visible_devices=torch.cuda.device_count(),total_memory=p.total_memory,environment=collect_environment_diagnostics()),indent=2))"
    run("target", [str(python), "-c", target_code, str(evidence / "target.json")], required=True)
    run(
        "checkpoint-verify",
        [
            str(python),
            "-m",
            "scripts.verify_owned_checkpoint",
            "--output",
            str(evidence / "environment-manifest.json"),
        ],
        required=True,
    )
    setup_ok = True
except BaseException:
    (evidence / "setup-failure.txt").write_text(traceback.format_exc())
    print("Setup incomplete. Run the final cell to download its logs.")

In [ ]:
checks_ok = False
try:
    assert setup_ok
    run(
        "cpu-component-checks",
        [
            str(python),
            "-m",
            "pytest",
            "tests/unit/test_checkpoint_diagnostic.py",
            "tests/unit/test_independent_components.py",
            "tests/unit/test_independent_decoder_math.py",
            "-q",
            "--junitxml=" + str(evidence / "cpu-components.xml"),
        ],
        required=True,
    )
    checks_ok = True
except BaseException:
    (evidence / "cpu-check-failure.txt").write_text(traceback.format_exc())
    print("CPU checks incomplete; run the export cell.")

In [ ]:
try:
    assert checks_ok
    run(
        "checkpoint-trajectory",
        [
            str(python),
            "-m",
            "scripts.check_checkpoint_trajectory",
            "--output-dir",
            str(evidence / "checkpoint-trajectory"),
        ],
        required=True,
    )
    result = json.loads((evidence / "checkpoint-trajectory/audited/report.json").read_text())
    print("Recorded cached steps:", len(result["rows"]))
    print("Component checks:", len(result["components"]))
    print("Instrumentation exact:", all(r["instrumentation_logits_exact"] for r in result["rows"]))
    print("Final step:", result["rows"][-1])
except BaseException:
    (evidence / "trajectory-failure.txt").write_text(traceback.format_exc())
    print("Run incomplete; export its partial evidence using the final cell.")

In [ ]:
try:
    (evidence / "source-after.json").write_text(json.dumps(verify_source(), indent=2))
except BaseException:
    (evidence / "source-check-failure.txt").write_text(traceback.format_exc())
(evidence / "commands.json").write_text(json.dumps(commands, indent=2))
(evidence / "notebook-run.json").write_text(
    json.dumps(
        {
            "snapshot_sha256": SNAPSHOT_SHA256,
            "base_revision": identity["base_revision"],
            "scope": "Real-checkpoint component and 32-step cached diagnostic; not accuracy or quality acceptance",
            "environment_overrides": {
                k: environment[k]
                for k in (
                    "CUDA_VISIBLE_DEVICES",
                    "CUBLAS_WORKSPACE_CONFIG",
                    "OMP_NUM_THREADS",
                    "MKL_NUM_THREADS",
                )
            },
        },
        indent=2,
    )
)
files = {
    p.relative_to(evidence).as_posix(): hashlib.sha256(p.read_bytes()).hexdigest()
    for p in evidence.rglob("*")
    if p.is_file()
}
(evidence / "checksums.json").write_text(json.dumps(files, indent=2))
archive_path = working / (run_root.name + ".zip")
with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    archive.writestr("source-snapshot.zip", payload)
    for p in evidence.rglob("*"):
        if p.is_file():
            archive.write(p, "evidence/" + p.relative_to(evidence).as_posix())
print("Download and send back:", archive_path.name)
print("If the link fails, download the ZIP from Kaggle's Output file panel.")
display(
    FileLink(
        str(archive_path.relative_to(Path.cwd()))
        if archive_path.is_relative_to(Path.cwd())
        else str(archive_path)
    )
)